# Tool Calling

In [1]:
import dotenv
from agents import Agent, ModelSettings, Runner, function_tool, trace

dotenv.load_dotenv()

True

Create a static calorie table that we can use as a tool:

In [4]:
def get_food_calories(food_item: str) -> str:
    """
    Get calorie information for common foods to help with nutrition tracking.

    Args:
        food_item: Name of the food (e.g., "apple", "banana")

    Returns:
        Calorie information per standard serving
    """
    # Simple calorie database - in real world, you'd use USDA API
    calorie_data = {
        "apple": "80 calories per medium apple (182g)",
        "banana": "105 calories per medium banana (118g)",
        "broccoli": "25 calories per 1 cup chopped (91g)",
        "almonds": "164 calories per 1oz (28g) or about 23 nuts",
    }

    food_key = food_item.lower()
    if food_key in calorie_data:
        return f"{food_item.title()}: {calorie_data[food_key]}"
    else:
        return f"I don't have calorie data for {food_item} in my database. Try common foods like apple, chicken breast, or rice."

In [3]:
def estimate_daily_water_intake(weight_kg: float) -> str:
    """Estimate a rough daily fluid intake from body weight, in kilograms."""
    if weight_kg <= 0:
        return "Weight must be greater than 0 kg."

    liters = weight_kg * 0.035
    cups = round(liters * 1000 / 240)
    return (
        f"A rough estimate for {weight_kg:g} kg is {liters:.1f} L of fluids per day "
        f"(about {cups} US cups). This is a general estimate; needs vary with "
        "activity, climate, and health conditions."
    )


hydration_agent = Agent(
    name="Hydration Assistant",
    instructions="""
    Help users estimate their daily fluid intake. Ask for their weight in kilograms
    if it is not provided, then use the estimate_daily_water_intake tool. Keep the
    answer concise and explain that the result is only a general estimate.
    """,
    tools=[function_tool(estimate_daily_water_intake)],
)

print(estimate_daily_water_intake(70))


A rough estimate for 70 kg is 2.5 L of fluids per day (about 10 US cups). This is a general estimate; needs vary with activity, climate, and health conditions.


Test the calorie lookup directly before registering it as an agent tool:

In [5]:
get_food_calories('apple')

'Apple: 80 calories per medium apple (182g)'

In [6]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information.
    You give concise answers.
    """,
    tools=[function_tool(get_food_calories)],
)

In [7]:
with trace("Nutrition Assistant with tools"):
    result = await Runner.run(
        calorie_agent, "How many calories are in total in a banana and an apple?"
    )
    print(result.final_output)

About 185 calories (medium banana ~105 kcal + medium apple ~80 kcal).


Enforce tools use:

In [ ]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information.
    You give concise answers.
    """,
    tools=[function_tool(get_food_calories)],
    model_settings=ModelSettings(tool_choice="get_food_calories"),
)

with trace("Nutrition Assistant with tools enforced"):
    result = await Runner.run(
        calorie_agent, "How many calories are in total in a banana and an apple?"
    )
    print(result.final_output)

About 185 calories in total (105 kcal for the medium banana + 80 kcal for the medium apple).
